In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS datasheet_project.silver")
print("✅ Silver schema ready")

In [0]:
%skip
%sql
SELECT 
  file_name,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Extract these specs from this datasheet as JSON with keys: ',
      'chip_name, manufacturer, architecture, flash_kb, ram_kb, max_clock_mhz, ',
      'voltage_min, voltage_max, pin_count, operating_temp_min, operating_temp_max. ',
      'If a value is not found, use null. Return ONLY the JSON, no other text. ',
      'Datasheet text: ',
      SUBSTRING(raw_text, 1, 8000)
    )
  ) AS extracted_specs
FROM datasheet_project.bronze.datasheets_raw
WHERE file_name = 'atmega328pb.pdf'

In [0]:
%sql
CREATE OR REPLACE TABLE datasheet_project.silver.datasheets_extracted AS
SELECT 
  file_name,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Extract these specs from this datasheet as JSON with keys: ',
      'chip_name, manufacturer, architecture, flash_kb, ram_kb, max_clock_mhz, ',
      'voltage_min, voltage_max, pin_count, operating_temp_min, operating_temp_max. ',
      'If a value is not found, use null. Return ONLY the JSON, no markdown, no other text. ',
      'Datasheet text: ',
      SUBSTRING(raw_text, 1, 8000)
    )
  ) AS extracted_json,
  current_timestamp() AS extracted_at
FROM datasheet_project.bronze.datasheets_raw

In [0]:
df = spark.table("datasheet_project.silver.datasheets_extracted")
df.show(truncate=False)

In [0]:
%sql
CREATE OR REPLACE TABLE datasheet_project.silver.datasheets_clean AS
WITH cleaned AS (
  SELECT 
    file_name,
    regexp_replace(
      regexp_replace(extracted_json, '^```(json)?\\s*', ''),
      '```\\s*$', ''
    ) AS json_text,
    extracted_at
  FROM datasheet_project.silver.datasheets_extracted
),
parsed AS (
  SELECT
    file_name,
    from_json(json_text, 'struct<
      chip_name:string,
      manufacturer:string,
      architecture:string,
      flash_kb:double,
      ram_kb:double,
      max_clock_mhz:double,
      voltage_min:double,
      voltage_max:double,
      pin_count:int,
      operating_temp_min:int,
      operating_temp_max:int
    >') AS specs,
    extracted_at
  FROM cleaned
)
SELECT
  file_name,
  specs.chip_name,
  specs.manufacturer,
  specs.architecture,
  specs.flash_kb,
  specs.ram_kb,
  specs.max_clock_mhz,
  specs.voltage_min,
  specs.voltage_max,
  specs.pin_count,
  specs.operating_temp_min,
  specs.operating_temp_max,
  extracted_at
FROM parsed

In [0]:
df = spark.table("datasheet_project.silver.datasheets_clean")
display(df)